In [3]:
!pip install pyannote.audio pydub torchaudio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 898.7/898.7 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00

In [20]:
import torch
from pyannote.audio import Pipeline
from pydub import AudioSegment
import os
from pyannote.audio import Pipeline
import torch

HF_TOKEN = ""

AUDIO_PATH = "test.ogg"
OUTPUT_DIR = "speaker_segments"

os.makedirs(OUTPUT_DIR, exist_ok=True)

pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", use_auth_token=HF_TOKEN).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

diarization = pipeline(AUDIO_PATH)

print("\nРезультат диаризации:")
for turn, _, speaker in diarization.itertracks(yield_label=True):
    print(f"Спикер {speaker}: {turn.start:.1f}s – {turn.end:.1f}s")

audio = AudioSegment.from_file(AUDIO_PATH)

for segment in diarization.itertracks(yield_label=True):
    turn, _, speaker = segment
    start_ms = int(turn.start * 1000)
    end_ms = int(turn.end * 1000)

    segment = audio[start_ms:end_ms]

    output_path = os.path.join(
        OUTPUT_DIR,
        f"speaker_{speaker}_{turn.start:.1f}s-{turn.end:.1f}s.wav"
    )
    segment.export(output_path, format="wav")
    print(f"Сохранено: {output_path}")

print("\nГотово! Сегменты сохранены в папку:", OUTPUT_DIR)

/usr/local/lib/python3.11/dist-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  std = sequences.std(dim=-1, correction=1)
/usr/local/lib/python3.11/dist-packages/torchaudio/_backend/soundfile_backend.py:71: UserWarning: The OPUS subtype is unknown to TorchAudio. As a result, the bits_per_sample attribute will be set to 0. If you are seeing this warning, please report by opening an issue on github (after checking for existing/closed ones). You may otherwise ignore this warning.
  warnings.warn(



Результат диаризации:
Спикер SPEAKER_00: 0.1s – 0.8s
Спикер SPEAKER_00: 2.9s – 4.1s
Спикер SPEAKER_00: 5.5s – 6.8s
Спикер SPEAKER_00: 7.5s – 7.9s
Спикер SPEAKER_00: 9.0s – 9.6s
Сохранено: speaker_segments/speaker_SPEAKER_00_0.1s-0.8s.wav
Сохранено: speaker_segments/speaker_SPEAKER_00_2.9s-4.1s.wav
Сохранено: speaker_segments/speaker_SPEAKER_00_5.5s-6.8s.wav
Сохранено: speaker_segments/speaker_SPEAKER_00_7.5s-7.9s.wav
Сохранено: speaker_segments/speaker_SPEAKER_00_9.0s-9.6s.wav

Готово! Сегменты сохранены в папку: speaker_segments
